# 🏠 Apartment Hunter : estimation du prix d'un bien immobilier

*Projet La Plateforme, B3 IA*

## Contexte

Fixer le prix d'un bien immobilier est difficile : il dépend de nombreuses caractéristiques (surface, nombre de pièces, état, localisation, vue, année de construction…) qui interagissent entre elles. Une estimation trop haute fait traîner la vente, une estimation trop basse fait perdre de l'argent au vendeur.

Nous utilisons le jeu de données **King County House Sales** (`kc_house_data.csv`) : **21 613 ventes** de maisons dans le comté de King (région de Seattle, États-Unis) entre mai 2014 et mai 2015. Chaque ligne décrit une maison vendue et son prix de vente (`price`), qui sera notre **variable cible**.

## Problématique

> **Peut-on estimer le prix d'un bien immobilier à partir de ses caractéristiques ?**

Il s'agit d'un problème de **régression supervisée** : prédire une valeur continue (le prix) à partir de variables explicatives. L'objectif final est de livrer un modèle exploitable via une application web Flask, dans laquelle un utilisateur saisit les caractéristiques d'un bien et obtient une estimation de son prix.

## Plan du notebook

1. **Imports et versions** : bibliothèques utilisées, versions affichées pour la reproductibilité
2. **Chargement et exploration des données** : structure, types, valeurs manquantes, distributions, valeurs aberrantes
3. **Séparation train / test** : effectuée *avant* tout traitement qui apprend des données, afin d'éviter toute fuite d'information (*data leakage*)
4. **Préparation des données** : nettoyage et création de variables, ajustés uniquement sur le jeu d'entraînement
5. **Sélection de features** : Boruta, puis *forward feature selection*
6. **Entraînement et comparaison de modèles** : régression linéaire / Ridge, Random Forest, Gradient Boosting, évalués avec MAE, RMSE et R²
7. **Optimisation par Grid Search** du meilleur modèle
8. **Évaluation finale sur le jeu de test et sauvegarde** du modèle avec `joblib`
9. **Conclusion** : réponse à la problématique, limites et pistes d'amélioration

> Pour la reproductibilité, toutes les opérations aléatoires utilisent `random_state=42`.

## 1. Imports et versions

On importe ici toutes les bibliothèques du notebook, regroupées par usage. Afficher leurs versions permet de reproduire exactement les résultats : un changement de version de scikit-learn, par exemple, peut modifier légèrement un modèle. Ces versions correspondent à celles figées dans `requirements.txt`.

In [ ]:
# --- Bibliothèques standard ---
import sys
import warnings
from importlib.metadata import version

# --- Manipulation des données ---
import numpy as np
import pandas as pd

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Préparation des données et validation ---
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# --- Sélection de features ---
from boruta import BorutaPy
from sklearn.feature_selection import SequentialFeatureSelector

# --- Modèles de régression ---
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# --- Métriques d'évaluation ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Sauvegarde du modèle ---
import joblib

# Graine aléatoire unique, réutilisée partout pour la reproductibilité
RANDOM_STATE = 42

# Options d'affichage : toutes les colonnes visibles, style graphique homogène
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# Affichage des versions (lues depuis les métadonnées des paquets installés)
print(f"Python       : {sys.version.split()[0]}")
for paquet in ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "Boruta", "joblib"]:
    print(f"{paquet:<13}: {version(paquet)}")